# Filtracja bilateralna

## Konwolucja obrazu z filtrem o zadanych współczynnikach

Splot (konwolucję) obrazu wejściowego $I$ z filtrem $\psi$ dla ustalonego punktu obrazu $\mathbf{x}$ można przedstawić następująco:

\begin{equation}
\hat{I}(\mathbf{x}) = \frac{1}{W_N}\sum_{\mathbf{p} \in \eta(\mathbf{x})} \psi(||\mathbf{p}-\mathbf{x}||)I(\mathbf{p})
\end{equation}

gdzie:
- $\hat{I}$ jest obrazem wynikowym (przefiltrowanym),
- $W_N = \sum_y \psi(y)$ jest parametrem normalizującym współczynniki filtra $\psi$,
- $||\cdot||$ jest odległością między punktami obrazu $\mathbf{x}$ i $\mathbf{p}$ według ustalonej metryki (np. norma $L_2$). Uwaga, proszę pamiętać, że zarówno $\mathbf{x}$, jak i $\mathbf{p}$ to współrzędne przestrzenne,
- $\eta(\mathbf{x})$ jest otoczeniem punktu $\mathbf{x}$.

Funkcja $\psi$ decyduje o charakterze filtracji. Dla filtru Gaussowskiego:

\begin{equation}
\psi(y) = G_{\delta_s}(y)
\end{equation}

gdzie: $G_{\delta_s}(y)$ jest funkcją Gaussa z parametrem skali $\delta_s$.

Opisaną powyżej filtrację realizowaliśmy w ramach ćwiczenia "Przetwarzanie wstępne. Filtracja kontekstowa."

## Filtracja bilateralna

Wadą klasycznego splotu jest brak adaptacji współczynników filtra do lokalnego otoczenia $\eta(\mathbf{x})$ filtrowanego punktu $\mathbf{x}$.
Oznacza to wykorzystanie tych samych współczynników filtra $\psi$ niezależnie od tego czy otoczenie jest względnie jednorodne lub zawiera krawędzie obiektów (w tym przypadku dochodzi do "rozmywania" krawędzi).
Filtracja bilateralna uwzględnia lokalne otoczenie filtrowanego punktu, w ten sposób, że parametry filtra zmieniają się w zależności od "wyglądu" otocznia.


Współczynniki filtra obliczane są na podstawie odległości filtrowanego punktu $\mathbf{x}$ od każdego punktu otoczenia $\mathbf{p}$ w dziedzinie przestrzennej obrazu (tak jak przy typowym filtrze np. Gaussa) oraz odległości punktów w przeciwdziedzinie obrazu (np. na podstawie różnicy w jasności pikseli dla obrazu w odcieniach szarości):

\begin{equation}
\hat{I}(\mathbf{x}) = \frac{1}{W_N}\sum_{\mathbf{p} \in \eta(\mathbf{x})} \psi(||\mathbf{p}-\mathbf{x}||) \gamma(|I(\mathbf{p}) - I(\mathbf{x})|) I(\mathbf{p})
\end{equation}
gdzie:
- $W_N$ jest współczynnikiem normalizującym filtr,
- $\gamma$ jest funkcją odległości w przeciwdziedzinie obrazu, np. $\gamma(y)=\exp(-\frac{y^2}{2\delta_r^2})$
- parametr $\delta_r$ jest utożsamiany z poziomem szumu w obrazie i należy go dobrać w sposób empiryczny.

Proszę chwilę zastanowić się nad powyższym równaniem, w szczególności nad funkcją $\gamma$. Proszę wyznaczyć, jaka będzie wartość funkcji dla pikseli podobnych (różnica 0, 1, 2), a skrajnie różnych (255, 200).

##  Realizacja ćwiczenia

### Wczytanie danych

1. Wczytaj dane z pliku *MR_data.mat*. W tym celu wykorzystaj funkcję `loadmat` z pakietu scipy:
        from scipy.io import loadmat
        mat = loadmat('MR_data.mat')

2. Wczytany plik zawiera 5 obrazów: *I_noisefree*, *I_noisy1*, *I_noisy2*, *I_noisy3* oraz *I_noisy4*. Odczytać je można w następujący sposób:
        Input = mat['I_noisy1']

3.Wyświetl wybrany obraz z pliku *MR_data.mat*. Zagadka - co to za obrazowanie medyczne?

In [ ]:
import cv2
import os
import requests
from matplotlib import pyplot as plt
import numpy as np
from scipy import signal
from scipy.io import loadmat
import math

url = 'https://raw.githubusercontent.com/vision-agh/poc_sw/master/07_Bilateral/'

fileNames = ["MR_data.mat"]
for fileName in fileNames:
  if not os.path.exists(fileName):
      r = requests.get(url + fileName, allow_redirects=True)
      open(fileName, 'wb').write(r.content)

#TODO Samodzielna

mat = loadmat('MR_data.mat')
I_noisefree = mat['I_noisefree']
I_noisy1 = mat['I_noisy1']
I_noisy2 = mat['I_noisy2']
I_noisy3 = mat['I_noisy3']
I_noisy4 = mat['I_noisy4']

Images = {
   "I_noisefree" : I_noisefree, 
   "I_noisy1" : I_noisy1, 
   "I_noisy2" : I_noisy2, 
   "I_noisy3" : I_noisy3, 
   "I_noisy4" : I_noisy4
}

fig, ax = plt.subplots(1, 5, figsize=(15, 6))

ax[0].imshow(I_noisefree, cmap='gray')
ax[0].set_title("Obraz oryginalny")
ax[0].axis('off')

ax[1].imshow(I_noisy1, cmap='gray')
ax[1].set_title("I_noisy1")
ax[1].axis('off')

ax[2].imshow(I_noisy2, cmap='gray')
ax[2].set_title("I_noisy2")
ax[2].axis('off')

ax[3].imshow(I_noisy3, cmap='gray')
ax[3].set_title("I_noisy3")
ax[3].axis('off')

ax[4].imshow(I_noisy4, cmap='gray')
ax[4].set_title("I_noisy4")
ax[4].axis('off')

plt.show()

### "Klasyczna" konwolucja

1. Zdefiniuj parametry filtra Gaussowskiego: rozmiar okna i wariancję $\delta_S$.
2. Oblicz współczynniki filtra na podstawie zdefiniowanych parametrów (najprościej w ramach podwójnej pętli for).
2. Sprawdź ich poprawność i zwizualizuj filtr (tak jak w ćwiczeniu pt. "Przetwarzanie wstępne. Filtracja kontekstowa.").
3. Wykonaj kopię obrazu wejściowego: `IConv = Input.copy()`
4. Wykonaj podwójną pętlę po obrazie. Pomiń ramkę, dla której nie jest zdefiniowany kontekst o wybranej wielkości.
5. W każdej iteracji stwórz dwuwymiarową tablicę zawierającą aktualny kontekst.
6. Napisz funkcję, która będzie obliczała nową wartość piksela.
Argumentem tej funkcji są aktualnie przetwarzane okno i współczynniki filtra.
7. Obliczoną wartość przypisz do odpowiedniego piksela kopii obrazu wejściowego.
8. Wyświetl wynik filtracji.
9. Porównaj wynik z obrazem oryginalnym.

In [ ]:
#TODO Samodzielna

window_size = 5
sigma = 0.7

cases = {
    3 : (0.5, 0.7, 0.9),
    5 : (0.5, 0.7, 0.9),
    7 : (0.5, 0.7, 0.9)
}

def conv_classic(name, window_size, sigma):
    kernel = np.zeros((window_size, window_size))
    Input = Images[name]

    def calculate_filtr_factor(sigma, kernel, window_size):
        radius = window_size // 2
        for x in range(-radius, radius + 1):
            for y in range(-radius, radius + 1):
                exp_value = -(x**2 + y**2) / (2 * (sigma**2)) # wzór na rozkład Gaussa funkcja psi -> skupiamy się na pikselach w okolicy, które są blisko
                kernel[x + radius, y + radius] = np.exp(exp_value)

    calculate_filtr_factor(sigma, kernel, window_size)

    kernel = kernel / np.sum(kernel)

    radius = window_size // 2

    IConv = Input.copy().astype('float64')
    rows, cols = Input.shape

    def calculate_pixel(window, kernel):
        result = window * kernel
        new_val =  np.sum(result)
        return new_val

    for i in range(radius, rows - radius):
        for j in range(radius, cols - radius):
            window = Input[i-radius : i+radius+1, j-radius : j+radius+1]
            new_val = calculate_pixel(window, kernel)
            IConv[i, j] = new_val

    IConv = np.clip(IConv, 0, 255).astype('uint8')

    fig = plt.figure(figsize=(15, 5))
    
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    X = np.arange(-radius, radius + 1, 1)
    Y = np.arange(-radius, radius + 1, 1)
    X, Y = np.meshgrid(X, Y)
    ax1.plot_surface(X, Y, kernel)
    ax1.set_title(f"Kernel {window_size}x{window_size}, sigma={sigma}")

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.imshow(Input, cmap='gray')
    ax2.set_title(f"Oryginał: {name}")
    ax2.axis('off')

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.imshow(IConv, cmap='gray')
    ax3.set_title(f"Gauss ({window_size}x{window_size}) sigma: {sigma}")
    ax3.axis('off')

    plt.tight_layout()
    plt.show()

for name in Images:
    if name in ("I_noisefree", "I_noisy3", "I_noisy4") : continue
    for size in cases.keys():
        for sigma in cases[size]:
            conv_classic(name, size, sigma)

### Filtracja bilateralna

1. Zdefiniuj dodatkowy parametr: wariancję $\delta_R$.
3. Wykonaj kopię obrazu wejściowego: `IBilateral = Input.copy()`
4. Wykonaj podwójną pętlę po obrazie. Pomiń ramkę, dla której nie jest zdefiniowany kontekst o wybranej wielkości.
5. W każdej iteracji stwórz dwuwymiarową tablicę zawierającą aktualny kontekst.
6. Napisz funkcję, która będzie obliczała nową wartość piksela.
Argumentami funkcji są aktualnie przetwarzane okno, współczynniki filtra gausowskiego (takie same jak wcześniej) i wariancja $\delta_R$.
7. Oblicz odległość w przeciwdziedzinie (dla wartości pikseli).
8. Oblicz funkcję Gaussa dla obliczonych odległości z zadanym parametrem.
9. Wykonaj normalizację obliczonych współczynników.
10. Obliczoną wartość przypisz do odpowiedniego piksela kopii obrazu wejściowego.
11. Wyświetl wynik filtracji.
12. Porównaj wynik z obrazem oryginalnym.

In [ ]:

window_size = 5
sigma = 0.7

cases = {
    3 : (
        (0.5, 10),
        (0.5, 30),
        (0.5, 80)
    ),
    5 : (
        (1.0, 10),
        (1.0, 30),
        (1.0, 80)
    ),
    7 : (
        (2.0, 10),
        (2.0, 30),
        (2.0, 80)
    )
}

def conv_bilateral(name, window_size, sigma, sigma_r):
    kernel = np.zeros((window_size, window_size))
    Input = Images[name]

    def calculate_filtr_factor(sigma, kernel, window_size):
        radius = window_size // 2
        for x in range(-radius, radius + 1):
            for y in range(-radius, radius + 1):
                exp_value = -(x**2 + y**2) / (2 * (sigma**2)) # wzór na rozkład Gaussa funkcja psi?
                kernel[x + radius, y + radius] = np.exp(exp_value)

    calculate_filtr_factor(sigma, kernel, window_size)

    kernel = kernel / np.sum(kernel)

    radius = window_size // 2

    IBilateral = Input.copy().astype('float64')
    rows, cols = Input.shape

    def calculate_pixel(window, kernel, sigma_r):
        center_val = window[radius, radius] # I(x)
        diff = window - center_val # I(p) - I(x) podobieństwo
        kernel_range = np.exp(-(diff**2) / (2 * sigma_r**2)) # funkcja gamma wzór na rozkład Gaussa -> skupiamy się na pikselach o podobnej jasności
        kernel_combined = kernel * kernel_range # waga przestrzenna razy waga koloru (psi * gamma)
        norm_factor = np.sum(kernel_combined) 
        result = np.sum(window * kernel_combined) / norm_factor # normalizacja * I(p) / W wspołczynnik normalizujący
    
        return result

    for i in range(radius, rows - radius):
        for j in range(radius, cols - radius):
            window = Input[i-radius : i+radius+1, j-radius : j+radius+1]
            new_val = calculate_pixel(window, kernel, sigma_r)
            IBilateral[i, j] = new_val

    IBilateral = np.clip(IBilateral, 0, 255).astype('uint8')

    fig = plt.figure(figsize=(15, 5))
    
    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    X = np.arange(-radius, radius + 1, 1)
    Y = np.arange(-radius, radius + 1, 1)
    X, Y = np.meshgrid(X, Y)
    ax1.plot_surface(X, Y, kernel)
    ax1.set_title(f"Kernel {window_size}x{window_size}, sigma={sigma}, sigma_r={sigma_r}")

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.imshow(Input, cmap='gray')
    ax2.set_title(f"Oryginał: {name}")
    ax2.axis('off')

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.imshow(IBilateral, cmap='gray')
    ax3.set_title(f"Gauss ({window_size}x{window_size}) sigma: {sigma}, sigma_r={sigma_r}")
    ax3.axis('off')

    plt.tight_layout()
    plt.show()

for name in Images:
    if name in ("I_noisefree", "I_noisy3", "I_noisy4") : continue
    for size in cases.keys():
        for tuple_sigma in cases[size]:
            sigma, sigma_r = tuple_sigma
            conv_bilateral(name, size, sigma, sigma_r)